In [ ]:
# Conversion of words into tokens has several encoding methods as follows ,
# a. o200k_base	                gpt-5 
# b. cl100k_base                gpt-5-turbo, gpt-5, gpt-5, text-embedding-ada-002, text-embedding-3-small, text-embedding-3-large
# c. p50k_base                  Codex models, text-davinci-002, text-davinci-003
# d. r50k_base (or gpt2)        GPT-3 models like davinci

# tiktoken is OpenAI's fast tokenizer library for counting tokens before making API calls
# Different models use different encodings: o200k_base for GPT-4o models, cl100k_base for GPT-3.5/GPT-4
# Counting tokens helps you estimate API costs and ensure your prompts fit within context windows

import os
from getpass import getpass
import tiktoken
from openai import OpenAI

text = "India is great country"

enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(text)
print("length:", len(tokens))


In [ ]:
# Counting tokens for chat completions API calls
# ns are counted from messages may change from model to model.
# Use tiktoken.encoding_for_model() to automatically load the correct encoding for a given model name.
# Message in response API can have - role, content and name fields.


def count_tokens_by_models(msg, model="gpt-5"):         # msg is chat message. gpt-5 is default
    try:    
        enc = tiktoken.encoding_for_model(model)   #"Which tokenizer should I use for this model?"
    except KeyError:
        print("model not found, defaulting to o200k_base")
        enc = tiktoken.get_encoding("o200k_base")

    if model in {
        "gpt-5",
        "gpt-5-0314"
    }: 
        tokens_per_message = 3       #accounts for the tokens used to structure the message.For e.g. "role", "content"
        tokens_per_name = 1          #The 3 and 1 are overhead values.
    else:
        raise NotImplementedError(
            f"Token counting not implemented for {model}"
        )

num_tokens = 0
for message in msg:
    num_tokens += tokens_per_message
    for key, value in message.items():
        num_tokens += len(enc.encode(value))
        if key == "name":
            num_tokens += tokens_per_name
num_tokens += 3                 # every reply is primed with <|start|>assistant<|message|>
return num_tokens